# 01 - Estudo exploratório do UNSW-NB15

**Objetivo:** verificar qualidade, distribuição e riscos dos dados antes de treinar os modelos.

Este notebook deve responder:

1. os arquivos possuem as colunas esperadas?
2. como está a distribuição entre normal e ataque?
3. há valores ausentes, infinitos ou categorias raras?
4. treino e teste apresentam distribuições muito diferentes?
5. quais decisões de pré-processamento devem ser registradas no relatório?

> Execute o Jupyter a partir da raiz do repositório para que os imports de `src` funcionem.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPOSITORY_ROOT = Path.cwd()
if not (REPOSITORY_ROOT / "src").exists() and (REPOSITORY_ROOT.parent / "src").exists():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.constants import ATTACK_CATEGORY_COLUMN, TARGET_COLUMN
from src.data import infer_column_groups, load_official_splits, split_features_target

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

DATA_DIR = REPOSITORY_ROOT / "data/raw"
train_df, test_df = load_official_splits(DATA_DIR)
print("Raiz do repositório:", REPOSITORY_ROOT)
print("Treino:", train_df.shape)
print("Teste: ", test_df.shape)

## 1. Amostra e esquema

In [ ]:
display(train_df.head())

schema = pd.DataFrame({
    "dtype": train_df.dtypes.astype(str),
    "missing": train_df.isna().sum(),
    "missing_pct": train_df.isna().mean().mul(100),
    "unique": train_df.nunique(dropna=False),
}).sort_values(["missing_pct", "unique"], ascending=[False, True])
display(schema)

### Verificações importantes

- `id` deve ser removida do modelo;
- `attack_cat` pode ser usada para análise de erros, mas não como entrada da classificação binária;
- `label` é o alvo;
- strings como `-` podem ser categorias válidas, não necessariamente valores ausentes.

## 2. Distribuição das classes

In [ ]:
def class_summary(frame: pd.DataFrame, name: str) -> pd.DataFrame:
    counts = frame[TARGET_COLUMN].value_counts().sort_index()
    result = pd.DataFrame({"count": counts, "pct": counts / counts.sum() * 100})
    result.index = result.index.map({0: "Normal", 1: "Ataque"})
    result.index.name = name
    return result

train_class = class_summary(train_df, "Treino")
test_class = class_summary(test_df, "Teste")
display(train_class)
display(test_class)

fig, ax = plt.subplots(figsize=(7, 4))
comparison = pd.DataFrame({
    "Treino": train_df[TARGET_COLUMN].value_counts(normalize=True).sort_index(),
    "Teste": test_df[TARGET_COLUMN].value_counts(normalize=True).sort_index(),
})
comparison.index = ["Normal", "Ataque"]
comparison.plot(kind="bar", ax=ax)
ax.set_ylabel("Proporção")
ax.set_title("Distribuição binária: treino versus teste")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
plt.show()

## 3. Categorias de ataque

In [ ]:
if ATTACK_CATEGORY_COLUMN in train_df.columns:
    attack_counts = (
        train_df[ATTACK_CATEGORY_COLUMN]
        .fillna("Normal/ausente")
        .astype(str)
        .value_counts()
    )
    display(attack_counts.to_frame("count"))

    fig, ax = plt.subplots(figsize=(9, 5))
    attack_counts.sort_values().plot(kind="barh", ax=ax)
    ax.set_xlabel("Registros")
    ax.set_title("Categorias no conjunto oficial de treino")
    fig.tight_layout()
    plt.show()

## 4. Colunas categóricas e cardinalidade

In [ ]:
x_train_raw, _ = split_features_target(train_df)
categorical_columns, numeric_columns = infer_column_groups(x_train_raw)

print("Categóricas:", categorical_columns)
print("Numéricas:", len(numeric_columns))

category_report = pd.DataFrame({
    "train_unique": train_df[categorical_columns].nunique(dropna=False),
    "test_unique": test_df[categorical_columns].nunique(dropna=False),
    "train_missing": train_df[categorical_columns].isna().sum(),
    "test_missing": test_df[categorical_columns].isna().sum(),
})
display(category_report)

for column in categorical_columns:
    print(f"\n### {column}")
    display(train_df[column].fillna("<NA>").value_counts().head(15).to_frame("train_count"))

## 5. Valores ausentes, infinitos e duplicados

In [ ]:
missing_report = pd.DataFrame({
    "train_missing": train_df.isna().sum(),
    "train_pct": train_df.isna().mean() * 100,
    "test_missing": test_df.isna().sum(),
    "test_pct": test_df.isna().mean() * 100,
}).query("train_missing > 0 or test_missing > 0")
display(missing_report.sort_values("train_pct", ascending=False))

numeric_train = train_df.select_dtypes(include=np.number)
inf_counts = np.isinf(numeric_train).sum().sort_values(ascending=False)
display(inf_counts[inf_counts > 0].to_frame("infinite_values"))

print("Duplicados no treino:", train_df.duplicated().sum())
print("Duplicados no teste: ", test_df.duplicated().sum())

## 6. Estatísticas numéricas

In [ ]:
numeric_summary = train_df[numeric_columns].describe().T
numeric_summary["missing"] = train_df[numeric_columns].isna().sum()
display(numeric_summary)

variance = train_df[numeric_columns].var(numeric_only=True).sort_values()
print("Menores variâncias:")
display(variance.head(10).to_frame("variance"))

## 7. Diferença simples entre treino e teste

In [ ]:
train_means = train_df[numeric_columns].mean()
test_means = test_df[numeric_columns].mean()
pooled_scale = train_df[numeric_columns].std().replace(0, np.nan)
standardized_mean_difference = ((test_means - train_means) / pooled_scale).abs().sort_values(ascending=False)

display(standardized_mean_difference.head(15).to_frame("abs_standardized_mean_difference"))

fig, ax = plt.subplots(figsize=(9, 5))
standardized_mean_difference.head(15).sort_values().plot(kind="barh", ax=ax)
ax.set_xlabel("|média_teste - média_treino| / desvio_treino")
ax.set_title("Sinal exploratório de mudança de distribuição")
fig.tight_layout()
plt.show()

> Esta comparação não é um teste formal de drift. Ela serve para identificar colunas que merecem inspeção e discussão no relatório.

## 8. Correlação com o rótulo e entre atributos

In [ ]:
correlations = (
    train_df[numeric_columns + [TARGET_COLUMN]]
    .corr(numeric_only=True)[TARGET_COLUMN]
    .drop(TARGET_COLUMN)
    .sort_values(key=lambda series: series.abs(), ascending=False)
)
display(correlations.head(20).to_frame("correlation_with_label"))

selected = correlations.head(15).index.tolist()
correlation_matrix = train_df[selected].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(correlation_matrix, aspect="auto")
fig.colorbar(image, ax=ax)
ax.set_xticks(range(len(selected)), selected, rotation=90)
ax.set_yticks(range(len(selected)), selected)
ax.set_title("Correlação entre os 15 atributos mais associados ao rótulo")
fig.tight_layout()
plt.show()

## 9. Checklist antes do treinamento

In [ ]:
checks = {
    "treino_tem_duas_classes": train_df[TARGET_COLUMN].nunique() == 2,
    "teste_tem_duas_classes": test_df[TARGET_COLUMN].nunique() == 2,
    "mesmas_colunas": set(train_df.columns) == set(test_df.columns),
    "id_presente_para_remocao": "id" in train_df.columns,
    "attack_cat_presente_para_analise": ATTACK_CATEGORY_COLUMN in train_df.columns,
}
pd.Series(checks, name="ok").to_frame()

## 10. Conclusões a preencher após a execução

Registre aqui, com números:

- proporção de ataques em treino e teste;
- colunas com valores ausentes ou infinitos;
- categorias novas no teste;
- possíveis diferenças de distribuição;
- atributos suspeitos de vazamento;
- decisão sobre pesos de classe;
- decisão sobre remoção de duplicados;
- riscos que devem aparecer na seção **Lições Aprendidas**.

Depois, registre o experimento E00/E01 em `docs/diario_experimentos.md`.